In [ ]:
# LINEAR REGRESSION FIT

import numpy as np
import matplotlib.pyplot as plt


R = 8.314462618  # J/(mol K)

# Approximate data digitized from figure 6
# r0 units: mmol / (g_cat min)
# P_CH4 units: atm

P_CH4 = np.array([0.03, 0.10, 0.19, 0.43], dtype=float)

T_C = np.array([800.0, 850.0, 900.0, 950.0])
T_K = T_C + 273.15

# Rows = temperatures
# Columns = P_CH4 = [0.03, 0.10, 0.19, 0.43] atm

r0 = np.array([
    [0.035, 0.085, 0.145, 0.300],
    [0.200, 0.470, 0.580, 0.760],
    [0.620, 1.350, 1.800, 2.480],
    [1.570, 2.450, 3.450, 4.300],
], dtype=float)

#Linear regression function
def linear_regression(x, y):
    """Ordinary least-squares regression: y = slope*x + intercept."""
    coeff = np.polyfit(x, y, 1)
    slope, intercept = coeff[0], coeff[1]

    y_pred = slope * x + intercept
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)

    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return slope, intercept, y_pred, r_squared


n = 0.5
x = P_CH4 ** n

kp = np.zeros(len(T_K))
r2_pressure = np.zeros(len(T_K))

for i, temperature in enumerate(T_C):

    y = r0[i, :]

    kp_i = np.sum(x * y) / np.sum(x ** 2)
    y_pred = kp_i * x

    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)

    r_squared = 1.0 - ss_res / ss_tot

    kp[i] = kp_i
    r2_pressure[i] = r_squared

    print(
        f"T = {temperature:.0f} °C | "
        f"kp = {kp_i:.6g} | "
        f"R² = {r_squared:.6f}"
    )

print()

#Arrhenius regression
def arrhenius_regression(T_K, kp, label):

    x = 1.0 / T_K
    y = np.log(kp)

    slope, intercept, y_pred, r_squared = linear_regression(x, y)

    Ea_J_mol = -slope * R
    Ea_kJ_mol = Ea_J_mol / 1000.0
    A = np.exp(intercept)

    print("=" * 75)
    print(label)
    print("=" * 75)
    print(f"Slope                = {slope:.8e} K")
    print(f"Intercept ln(A)       = {intercept:.8f}")
    print(f"A                     = {A:.8e}")
    print(f"Ea                    = {Ea_kJ_mol:.4f} kJ/mol")
    print(f"R² Arrhenius          = {r_squared:.6f}")
    print()

    return {
        "slope": slope,
        "intercept": intercept,
        "A": A,
        "Ea_J_mol": Ea_J_mol,
        "Ea_kJ_mol": Ea_kJ_mol,
        "r_squared": r_squared,
        "x": x,
        "y": y,
        "y_pred": y_pred,
    }

arr_fixed = arrhenius_regression(
    T_K, kp,
    "ARRHENIUS FIT"
)

#Paper parameters
A_paper = 1.0e10
Ea_paper_J_mol = 210000.0

#r0 comparison
r0_pred_fixed = np.zeros_like(r0)
r0_pred_paper = np.zeros_like(r0)

for i in range(len(T_K)):

    r0_pred_fixed[i, :] = (
        arr_fixed["A"]
        * np.exp(-arr_fixed["Ea_J_mol"] / (R * T_K[i]))
        * P_CH4 ** n
    )

    r0_pred_paper[i, :] = (
        A_paper
        * np.exp(-Ea_paper_J_mol / (R * T_K[i]))
        * P_CH4 ** n
    )

#Plot

import matplotlib as mpl
import scienceplots

from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

plt.style.use(["science", "ieee"])

tick_font = 25.0
axis_label_font = 26.0
legend_font = 20.0
legend_title_font = 21.0

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": tick_font,
    "axes.labelsize": axis_label_font,
    "xtick.labelsize": tick_font,
    "ytick.labelsize": tick_font,
    "legend.fontsize": legend_font,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(14.5, 9.2))

colors = [
    "#5B6FFF",
    "#F0523A",
    "#00BF91",
    "#9B59B6"
]

for j, pressure in enumerate(P_CH4):

    ax.plot(
        T_C, r0[:, j],
        linestyle="None",
        marker="x", markersize=13,
        markeredgewidth=2.8,
        color=colors[j],
        zorder=4
    )

    ax.plot(
        T_C, r0_pred_fixed[:, j],
        linestyle="-", linewidth=3.1,
        marker="o", markersize=9,
        markerfacecolor="white",
        markeredgecolor=colors[j],
        markeredgewidth=2.2,
        color=colors[j],
        zorder=3
    )

    ax.plot(
        T_C, r0_pred_paper[:, j],
        linestyle=":", linewidth=3.1,
        marker="^", markersize=10,
        markerfacecolor="white",
        markeredgecolor=colors[j],
        markeredgewidth=2.0,
        color=colors[j],
        zorder=2
    )

ax.set_xlabel(
    r"Temperature / [$^\circ$C]",
    fontsize=axis_label_font,
    labelpad=12
)

ax.set_ylabel(
    r"$r_0$ / [$\mathrm{mmol}\,\cdot\,"
    r"\mathrm{g}_{\mathrm{cat}}^{\mbox{-}1}"
    r"\,\cdot\,\mathrm{min}^{\mbox{-}1}$]",
    fontsize=axis_label_font,
    labelpad=14
)

ax.set_ylim(bottom=0)
ax.set_xlim(np.min(T_C) - 10, np.max(T_C) + 10)

ax.set_xticks(T_C)
ax.xaxis.set_minor_locator(MultipleLocator(25))
ax.yaxis.set_minor_locator(MultipleLocator(0.5))

ax.tick_params(
    axis="x", which="major",
    direction="in", top=True, bottom=True,
    labelsize=tick_font, colors="black",
    length=7.0, width=1.25, pad=8
)

ax.tick_params(
    axis="y", which="major",
    direction="in", left=True, right=True,
    labelsize=tick_font, colors="black",
    length=7.0, width=1.25, pad=8
)

ax.tick_params(
    axis="x", which="minor",
    direction="in", top=True, bottom=True,
    colors="black", length=4.0, width=0.9
)

ax.tick_params(
    axis="y", which="minor",
    direction="in", left=True, right=True,
    colors="black", length=4.0, width=0.9
)

ax.grid(False)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.25)

#Legends
pressure_handles = [
    Line2D(
        [0], [0],
        color=colors[j],
        linewidth=3.5,
        label=rf"{pressure:.2f} atm"
    )
    for j, pressure in enumerate(P_CH4)
]

legend_pressure = ax.legend(
    handles=pressure_handles,
    title=r"$P_{\mathrm{CH_4}}$",
    loc="upper left",
    bbox_to_anchor=(0.025, 0.975),
    fontsize=legend_font,
    title_fontsize=legend_title_font,
    frameon=True,
    fancybox=False,
    framealpha=1.0,
    facecolor="white",
    edgecolor="0.45",
    handlelength=2.1,
    handletextpad=0.5,
    borderpad=0.45,
    labelspacing=0.25
)

legend_pressure.get_frame().set_linewidth(1.0)
ax.add_artist(legend_pressure)

type_handles = [
    Line2D(
        [0], [0],
        color="black",
        linestyle="None",
        marker="x",
        markersize=12,
        markeredgewidth=2.6,
        label="Dataset"
    ),
    Line2D(
        [0], [0],
        color="black",
        linestyle="-",
        linewidth=3.1,
        marker="o",
        markersize=9,
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=2.0,
        label="Refitted model"
    ),
    Line2D(
        [0], [0],
        color="black",
        linestyle=":",
        linewidth=3.1,
        marker="^",
        markersize=10,
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=1.9,
        label="Paper model"
    )
]

legend_type = ax.legend(
    handles=type_handles,
    title="Data source",
    loc="upper left",
    bbox_to_anchor=(0.025, 0.68),
    fontsize=legend_font,
    title_fontsize=legend_title_font,
    frameon=True,
    fancybox=False,
    framealpha=1.0,
    facecolor="white",
    edgecolor="0.45",
    handlelength=2.4,
    handletextpad=0.7,
    borderpad=0.55,
    labelspacing=0.40
)

legend_type.get_frame().set_linewidth(1.0)

fig.subplots_adjust(
    left=0.13,
    right=0.985,
    bottom=0.15,
    top=0.985
)

fig.savefig(
    "r0_dataset_refitted_paper_model.pdf",
    format="pdf"
)

plt.show()
